# 01 – Szenario: Zustand erstellen, analysieren, speichern

Ein registrierter Zustand wird aus seiner Excel-Verteilung gebildet, geprüft, optional energetisch bewertet und optional gespeichert. **Standard: nichts wird berechnet oder gespeichert**; nur die Eingaben werden angezeigt bzw. ein gespeichertes Ergebnis geladen.

Ablauf: Szenario wählen (Steuerung) → optional Prioren aus globalen Zielen ableiten → Verteilungsdatei und Flächenannahmen anzeigen und prüfen → Zustand ausdrücklich bilden, Bilanzkontrollen → Eigenschaften, optional Energieanalyse → optional speichern → gespeichertes Ergebnis laden und mit der Sitzung vergleichen.

## Start

- **Was:** Projektordner finden und Modell laden.
- **Eingaben:** `config/projekt.json`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** –
- **Sperrende Fehler:** Projektordner nicht gefunden.

In [1]:
# Einheitliche Startzelle: Projektordner suchen, aktuelle Modellmodule laden, Fassung prüfen, Projektpfade lesen.
import os, sys
from pathlib import Path
try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config' / 'projekt.json').is_file())
except StopIteration:
    raise FileNotFoundError('Projektordner nicht gefunden: Das Notebook muss im Ordner notebooks/ des Projekts liegen (config/projekt.json fehlt).') from None
MODELL = ROOT / 'modell'
if str(MODELL) not in sys.path:
    sys.path.insert(0, str(MODELL))
for _name in [n for n, m in list(sys.modules.items()) if os.path.normcase(str(getattr(m, '__file__', None) or '')).startswith(os.path.normcase(str(MODELL)))]:
    del sys.modules[_name]   # nach einer Aktualisierung immer die Module aus dem Projektordner laden (kein veralteter Kernelstand)
import fassung
print('Projektordner:', ROOT)
print(fassung.pruefe_notebook('01_Szenario', '1.6.3-AP6'))
import gebaeudesektor as gs
from IPython.display import display
PROJEKT = gs.lade_projekt(ROOT)
import prioren as pr

Projektordner: c:\Users\ma0026.NB-MA0026\Documents\peq-higrube\Berechnungen\gebaeudesektormodell\Gebaeudesektor_konsolidiert\Gebaeudesektor
Fassung 1.6.3-AP6 (Notebook und Modell passen zusammen)


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Steuerung

- **Was:** Auswahl des Szenarios und der Schritte.
- **Eingaben:** Szenario-ID aus `config/szenarien.json`
- **Rechnet oder liest:** –
- **Annahmen:** Alle Schalter stehen standardmäßig auf `False`.
- **Interpretation:** `ERSETZEN=True` ist nur nötig, wenn ein vorhandenes Ergebnis bewusst ersetzt werden soll.
- **Sperrende Fehler:** –

In [6]:
SZENARIO = 'Transition_2040_ambitioniert'      # registrierte Szenario-ID (siehe 00_Start)
PRIOREN_ABLEITEN = True       # nur Szenarien mit Angabe "prioren": leere Prior-Zellen ableiten, abgeleitete Datei schreiben
BERECHNEN = True              # Zustand aus der Excel-Verteilung bilden
ENERGIE_BERECHNEN = True      # energetische Auswertung (benötigt scenarios.parquet und pyarrow)
EXPORTIEREN = True            # Ergebnis nach results/szenarien/<SZENARIO>/ speichern
ERSETZEN = True               # vorhandenes Ergebnis ersetzen (nur bewusst setzen)

## Prioren aus globalen Zielen (optional)

- **Was:** Für Szenarien mit Angabe `prioren` im Profil: Status der abgeleiteten Verteilungsdatei; bei `PRIOREN_ABLEITEN=True` werden leere Prior-Zellen der Eingabedatei abgeleitet und die vollständige Datei geschrieben (nur wenn sie fehlt oder veraltet ist).
- **Eingaben:** Eingabedatei (`prioren.eingabe`, leere Zellen erlaubt) mit ZIEL-Zeilen, gespeicherte Basis (`prioren.basis`, z. B. `Bestand_2024`), `uebergaenge.json`, bei registriertem Paar Basis → Szenario Neubau/Abriss und Freigaben aus `Bestandsentwicklung.xlsx`
- **Rechnet oder liest:** liest den Status; rechnet und schreibt nur bei `PRIOREN_ABLEITEN=True` (globale Verschiebung mit LP-Machbarkeitsprüfung und iterativem proportionalem Anpassen, wenige Sekunden; keine Flussrekonstruktion)
- **Annahmen:** Eingegebene Werte haben Vorrang. Leere Zellen: Herkunftsmischung der Basisflächen nach der globalen Verschiebung (nur Verbleib, Regelübergänge, freigegebener Neubau/Abriss, WP nur in WP-Qualitäten), Geometrie/Speichermasse/Flexibilität aus der Basis. Die Kalibrierung auf die ZIEL-Zeilen erfolgt danach unverändert im Modell.
- **Interpretation:** Blatt Herkunft der abgeleiteten Datei zeigt je Zelle eingegeben/abgeleitet/Basis. Nullzellen ohne zulässige Herkunft sind gewollt. Die abgeleitete Datei nicht von Hand ändern (Status VERALTET).
- **Sperrende Fehler:** Ziele mit den Regeln/Freigaben nicht erreichbar (Fehlmengen werden genannt), Basis nicht gespeichert, NGF-Änderung ohne Bruttovorgaben, abgeleitete Datei fehlt/veraltet bei `BERECHNEN=True`

In [3]:
prior_status = pr.status(SZENARIO, PROJEKT)
if prior_status['status'] == pr.NICHT_KONFIGURIERT:
    print('Szenario ohne Prior-Ableitung: die registrierte Excel-Verteilung wird direkt verwendet.')
else:
    if PRIOREN_ABLEITEN:
        vorher, datei, ableitung = pr.ableiten(SZENARIO, PROJEKT)
        if ableitung is None:
            print('Abgeleitete Datei ist aktuell; keine Neuableitung.')
        else:
            print('Abgeleitet und geschrieben:', datei.relative_to(ROOT).as_posix(), '| vorher:', vorher['status'])
            display(pr.protokolltabelle(ableitung))
            display(pr.herkunftsuebersicht(ableitung['herkunft']))
        prior_status = pr.status(SZENARIO, PROJEKT)
    print('Prior-Ableitung:', prior_status['status'], '|', '; '.join(prior_status['gruende']) or 'passt zu Eingabe, Basis, Regeln und Bestandsentwicklung')
    if prior_status['status'] != pr.AKTUELL:
        if BERECHNEN:
            raise RuntimeError('Abgeleitete Verteilung fehlt oder ist veraltet: PRIOREN_ABLEITEN=True setzen und erneut ausführen.')
        print('Hinweis: vor BERECHNEN=True mit PRIOREN_ABLEITEN=True neu ableiten.')

Abgeleitet und geschrieben: data/eingaben/verteilungen/abgeleitet/Verteilung_Transition_2040_ambitioniert.xlsx | vorher: FEHLT


,Angabe,Wert
0,methode,1.0
1,szenario,Transition_2040_ambitioniert
2,eingabe_datei,data/eingaben/verteilungen/Verteilung_Transiti...
3,eingabe_sha256,4c728cd1f86e50b05dd488c896f6e955baeb1097e61ba2...
4,basis_szenario,Bestand_2024
5,basis_zustand_id,6d9c1ee883204a042747bf69783d52e4d51ced7eafdacd...
6,basis_verteilung_sha256,40b3ee4b41737e0a3a809b2da3b0a03a362039f5c64be4...
7,regeln_datei,config/flaechenstroeme/uebergaenge.json
8,regeln_sha256,7382178f2c086e2f9b6726be332b73a681b87653635470...
9,rekonstruktion,Bestand_2024__Transition_2040_ambitioniert


,Blatt,Herkunft,Zellen,Nullzellen
0,Flexibilitaet,Basis,18,8
1,Geometrie,eingegeben,700,619
2,PV,abgeleitet,40,0
3,Qualitaet,abgeleitet,81,58
4,Speichermasse,eingegeben,6,0
5,Versorgung,Basis (Qualität ohne Fläche),180,80
6,Versorgung,abgeleitet,630,330


Prior-Ableitung: AKTUELL | passt zu Eingabe, Basis, Regeln und Bestandsentwicklung


## Eingaben

- **Was:** Szenarioprofil und verwendete Eingabedateien mit Prüfsumme.
- **Eingaben:** `config/szenarien.json`, Excel-Verteilung, `annahmen.json`, Statistik, Katalog
- **Rechnet oder liest:** liest und prüft die Eingaben (keine Zustandsbildung)
- **Annahmen:** Excel ist die fachliche Quelle für Verteilungen und Ziele.
- **Interpretation:** Dateipfade sind projektrelativ.
- **Sperrende Fehler:** fehlende Datei, ungültige Verteilung, WP-Anteil in ausgeschlossener Qualität

In [4]:
eingabe = gs.lade_szenario(SZENARIO, PROJEKT)
profil, quellen = gs.zeige_eingaben(eingabe)
display(profil); display(quellen)

,Angabe,Wert
0,beschreibung,"Flächenentwicklung wie Wachstum, hohe Sanierun..."
1,typ,Zukunftszustand
2,jahr,2040
3,statistikjahr,2024
4,verteilungen,data/eingaben/verteilungen/abgeleitet/Verteilu...
5,prioren,{'eingabe': 'data/eingaben/verteilungen/Vertei...
6,ziel_ngf_m2,950000000
7,ngf_zu_bgf_faktor,1.25
8,parquet_batch,None


,Quelle,Datei,Nachweis
0,basis,data/eingaben/annahmen.json,f0ae98a701f2d26069214d5a31026ea2e215c2227d20a9...
1,statistik,data/referenz/statistik/Statistik_Austria_2024...,2354690253deb7ad3b808b8ed57941389cce35cea53351...
2,katalog,data/referenz/typologien_43200.csv,bddc016ccd2a3dd41e9bfac541db7790507e220516363a...
3,excel,data/eingaben/verteilungen/abgeleitet/Verteilu...,0dba58a6e7003145a329f82ae6ab53ab0f345daa7afc3f...
4,szenarioregister,config/szenarien.json,7f101d8b1f129751ee1b40ce5fdfd3ccb7299ab88343a8...
5,parquet,data/simulation/scenarios.parquet,"schwach (Pfad/Größe/Zeitstempel), kein Inhalts..."


## Zustand bilden und Bilanzkontrollen

- **Was:** Bildet die NGF-Verteilung über den vollständigen Typenkatalog und zeigt die Bilanzkontrollen.
- **Eingaben:** geprüfte Eingaben
- **Rechnet oder liest:** rechnet (nur bei `BERECHNEN=True`)
- **Annahmen:** Kalibrierung und Regeln wie im Modul dokumentiert; keine Optimierung über Zielvorgaben.
- **Interpretation:** Kontrollen mit Status OK: Summen, Randverteilungen und Ausschlüsse sind eingehalten.
- **Sperrende Fehler:** Kontrolle mit Status FEHLER, negative Flächen, unvollständiger Katalog

In [5]:
zustand = bewertung = None
if BERECHNEN:
    zustand = gs.erstelle_zustand(eingabe)
    print(zustand['name'], zustand['jahr'], f"{zustand['verteilung'].NGF_m2.sum():,.0f} m² NGF", 'Zustand-ID', zustand['zustand_id'][:16])
    display(gs.bilanzkontrollen(zustand))
else:
    print('BERECHNEN=False: kein Zustand gebildet.')

Transition_2040_ambitioniert 2040 950,000,000 m² NGF Zustand-ID 7efe32919bdba57d


,Pruefung,Ist,Soll,Status
0,Geometrie Gebäude für Gemeinschaften/1 Geschoß...,1.000000e+00,1.000000e+00,OK
1,Geometrie Gebäude für Gemeinschaften/2 Geschoß...,1.000000e+00,1.000000e+00,OK
2,Geometrie Gebäude für Gemeinschaften/3 Geschoß...,1.000000e+00,1.000000e+00,OK
3,Geometrie Gebäude für Gemeinschaften/4 Geschoß...,1.000000e+00,1.000000e+00,OK
4,Geometrie Gebäude für Gemeinschaften/5 Geschoß...,1.000000e+00,1.000000e+00,OK
...,...,...,...,...
227,Masse je Nutzung,0.000000e+00,0.000000e+00,OK
228,Katalogumfang,4.320000e+04,4.320000e+04,OK
229,Übergabe-NGF,9.500000e+08,9.500000e+08,OK
230,Übergabe-BGF,1.187500e+09,1.187500e+09,OK


## Erreichbarkeit (AP6.3)

- **Was:** Prüft den eben gebildeten Zustand gegen die gespeicherten Gegenzustände aller registrierten Flächenstrompaare, an denen das Szenario beteiligt ist (Blockdiagnose wie in 03, mit Neubau/Abriss aus `Bestandsentwicklung.xlsx`).
- **Eingaben:** Zustand aus dieser Sitzung, gespeicherte Gegenzustände unter `results/szenarien/`, `rekonstruktionen.json`, Regeldatei des Paars, Bestandsentwicklung
- **Rechnet oder liest:** liest nur (nur wenn ein Zustand gebildet wurde): notwendige Bedingungen je Block und Merkmalskombination, kein Solver, keine Optimierung, kein Schreiben; etwa 10 s je Paar
- **Annahmen:** Geprüft werden notwendige, nicht hinreichende Bedingungen; Machbarkeit weist erst die Rekonstruktion in 03 nach.
- **Interpretation:** KEINE_KONFLIKTE: Paar kann in 03 rekonstruiert werden. KONFLIKTE: Zustand ist mit den Regeln und Freigaben voraussichtlich nicht erreichbar (Tabelle zeigt Block, Kombination und Defizit; Prioren, Ziele, Regeln oder Freigaben anpassen). GEGENZUSTAND_FEHLT: Gegenzustand zuerst speichern. Speichern bleibt möglich (Warnung).
- **Sperrende Fehler:** keine (nur Anzeige; Fehler je Paar in Spalte Meldung)

In [7]:
erreichbarkeit = None
if zustand is not None:
    erreichbarkeit, konflikte = pr.erreichbarkeit(zustand, PROJEKT)
    if erreichbarkeit.empty:
        print('Kein registriertes Flächenstrompaar mit', SZENARIO)
    else:
        display(erreichbarkeit)
        for rek_id, tabelle in konflikte.items():
            if len(tabelle):
                print(rek_id, '– größte Konflikte:')
                display(tabelle.head(15))

,Rekonstruktion,Rolle,Gegenzustand,Status,Bilanzstatus,Konflikte,Bloecke,Defizit_max_m2,Grundlage,Meldung
0,Bestand_2024__Transition_2040_ambitioniert,B,Bestand_2024,KEINE_KONFLIKTE,BILANZ_OK,0,0,0.0,"mit Neubau/Abriss (Bestandsentwicklung, Status...",notwendige Bedingungen erfüllt; Rekonstruktion...


## Eigenschaften und Energie

- **Was:** Eigenschaftsverteilungen; energetische Kennwerte nur bei `ENERGIE_BERECHNEN=True`.
- **Eingaben:** Zustand; optional `data/simulation/scenarios.parquet`
- **Rechnet oder liest:** rechnet (nur bei `BERECHNEN=True`)
- **Annahmen:** Bilanzdefinitionen unverändert; fehlende Kennwerte ergeben keine vollständigen Summen.
- **Interpretation:** Energiestatus „Nicht ausgeführt“ bedeutet: nur Flächenanalyse.
- **Sperrende Fehler:** Parquet fehlt bei angeforderter Energieanalyse

In [8]:
if zustand is not None:
    bewertung = gs.analysiere_zustand(zustand, eingabe, energie=ENERGIE_BERECHNEN)
    gs.zeige_ergebnisse(zustand, bewertung)
    display(gs.kennwerttabelle(zustand, bewertung).head(30))

Haushalts-Sollwerte2024 werden für dieses Szenariojahr nicht angewendet.


,Auspraegung,NGF_m2,BGF_m2,Belegte_Typen,NGF_Anteil_pct,Merkmal
0,EFH_1G,5.137206e+07,6.421508e+07,816,5.407586,Geometrie
1,GWB_7G,7.332863e+07,9.166078e+07,816,7.718803,Geometrie
2,EFH_2G,1.328859e+08,1.661074e+08,816,13.987994,Geometrie
3,NiWo_2G,1.158054e+08,1.447567e+08,816,12.190040,Geometrie
4,NiWo_3G,1.533035e+08,1.916294e+08,816,16.137211,Geometrie
5,RH_2G,1.672463e+08,2.090579e+08,816,17.604875,Geometrie
6,MFH_2G,8.510961e+07,1.063870e+08,816,8.958906,Geometrie
7,MFH_4G,5.854559e+07,7.318199e+07,816,6.162694,Geometrie
8,NiWo_1G,7.699229e+07,9.624036e+07,816,8.104451,Geometrie
9,GWB_5G,3.541069e+07,4.426336e+07,816,3.727441,Geometrie


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




Energie: Ausgeführt


,Szenario_ID,Zustand_ID,Jahr,Bereich,Gruppe,Kennwert,Kategorie,Wert,Einheit,Bezugsflaeche,Abdeckung_pct,Status,Vergleichsbasis_ID
0,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Gesamt,Fläche,NGF,,9.500000e+08,m²,NGF,100.0,OK,F-507e7b1fc41dcd82
1,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Gesamt,Fläche,BGF,,1.187500e+09,m²,BGF,100.0,OK,F-507e7b1fc41dcd82
2,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Gesamt,Typen,Positiv belegte Typen,,8.160000e+03,Anzahl,,100.0,OK,F-507e7b1fc41dcd82
3,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Haushalte,Fläche,NGF,,6.038988e+08,m²,NGF,100.0,OK,F-507e7b1fc41dcd82
4,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Haushalte,Fläche,BGF,,7.548735e+08,m²,BGF,100.0,OK,F-507e7b1fc41dcd82
5,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Haushalte,Typen,Positiv belegte Typen,,5.712000e+03,Anzahl,,100.0,OK,F-507e7b1fc41dcd82
6,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Nichtwohnen,Fläche,NGF,,3.461012e+08,m²,NGF,100.0,OK,F-507e7b1fc41dcd82
7,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Nichtwohnen,Fläche,BGF,,4.326265e+08,m²,BGF,100.0,OK,F-507e7b1fc41dcd82
8,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Nichtwohnen,Typen,Positiv belegte Typen,,2.448000e+03,Anzahl,,100.0,OK,F-507e7b1fc41dcd82
9,Transition_2040_ambitioniert,7efe32919bdba57d4c0e4f085aa6e06296d2bceac9ce2c...,2040,Gesamt,Geometrie,NGF,EFH_1G,5.137206e+07,m²,NGF,100.0,OK,F-507e7b1fc41dcd82


## Speichern

- **Was:** Speichert Zustand, Kennwerte und Laufprotokoll geprüft (erst in Arbeitsordner, dann veröffentlicht).
- **Eingaben:** Zustand und Bewertung aus dieser Sitzung
- **Rechnet oder liest:** schreibt nur bei `EXPORTIEREN=True`
- **Annahmen:** Vorhandene Ergebnisse sind ohne `ERSETZEN=True` geschützt.
- **Interpretation:** Laufprotokoll enthält Zustand-ID, Prüfsummen und projektrelative Pfade. Meldet die Erreichbarkeitsprüfung Konflikte, erscheint vor dem Speichern eine Warnung.
- **Sperrende Fehler:** vorhandenes Ergebnis ohne Freigabe, Zustand nicht berechnet

In [9]:
if EXPORTIEREN:
    if zustand is None:
        raise RuntimeError('Zum Speichern zuerst BERECHNEN=True setzen.')
    if erreichbarkeit is not None and erreichbarkeit.Status.isin(['KONFLIKTE', 'FEHLER']).any():
        print('WARNUNG: Erreichbarkeitsprüfung meldet Konflikte für', ', '.join(erreichbarkeit.loc[erreichbarkeit.Status.isin(['KONFLIKTE', 'FEHLER']), 'Rekonstruktion']),
              '– Zustand wird trotzdem gespeichert; Rekonstruktion in 03 wird voraussichtlich NICHT_MACHBAR.')
    ausgabe = gs.speichere_szenario(zustand, bewertung, eingabe, ersetzen=ERSETZEN)
    print('Gespeichert:', ausgabe.relative_to(ROOT).as_posix())

Gespeichert: results/szenarien/Transition_2040_ambitioniert


## Gespeichertes Ergebnis laden

- **Was:** Lädt das gespeicherte Ergebnis des Szenarios und prüft dessen Integrität.
- **Eingaben:** `results/szenarien/<SZENARIO>/`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** Integrität OK: Ausgaben entsprechen den Prüfsummen im Laufprotokoll.
- **Sperrende Fehler:** Ergebnis fehlt oder Prüfsummen weichen ab

In [10]:
ordner = PROJEKT['ergebnisse']['szenarien'] / SZENARIO
if ordner.is_dir():
    gespeichert = gs.lade_gespeichertes_szenario(SZENARIO, PROJEKT)
    print('Integrität:', gespeichert['integritaet'], '| Energie:', gespeichert['energiestatus'], '| Zustand-ID', str(gespeichert['zustand_id'])[:16])
    if zustand is not None:
        print('Gleicher Zustand wie in dieser Sitzung:', gespeichert['zustand_id'] == zustand['zustand_id'])
else:
    print('Noch kein gespeichertes Ergebnis für', SZENARIO)

Integrität: OK (SHA-256 der Ausgaben geprüft) | Energie: Ausgeführt | Zustand-ID 7efe32919bdba57d
Gleicher Zustand wie in dieser Sitzung: True
